In [1]:
import os
import glob
import warnings
import pickle
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
import lightgbm as lgb
import xgboost as xgb

matplotlib.use("Agg")
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

try:
    import shap
except Exception:
    shap = None

try:
    from evidently.metric_preset import ClassificationPreset, DataDriftPreset
    from evidently.report import Report
except Exception:
    Report = None
    DataDriftPreset = None
    ClassificationPreset = None

try:
    import vectorbt as vbt
except Exception:
    vbt = None


CONFIG = {
    "stocks_path": "/kaggle/input/datasets/borismarjanovic/price-volume-data-for-all-us-stocks-etfs/Stocks",
    "etf_path": "/kaggle/input/datasets/borismarjanovic/price-volume-data-for-all-us-stocks-etfs/ETFs",
    "max_stock_files": 400,
    "max_etf_files": 50,
    "top_n_tickers": 24,
    "seq_len": 20,
    "batch_size": 256,
    "epochs": 20,
    "future_horizon": 1,
    "transaction_cost": 0.0005,
    "num_trials": 25,
    "random_seed": 42,
    "threshold_metric": "f1",
    "regression_trials": 12,
    "scaler_type": "standard",
    "output_root": "artifacts",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

np.random.seed(CONFIG["random_seed"])
torch.manual_seed(CONFIG["random_seed"])
if CONFIG["device"] == "cuda":
    torch.cuda.manual_seed_all(CONFIG["random_seed"])

print("Device:", CONFIG["device"])

OUTPUT_DIRS = {
    "root": Path(CONFIG["output_root"]),
    "models": Path(CONFIG["output_root"]) / "models",
    "plots": Path(CONFIG["output_root"]) / "plots",
    "reports": Path(CONFIG["output_root"]) / "reports",
    "predictions": Path(CONFIG["output_root"]) / "predictions",
    "monitoring": Path(CONFIG["output_root"]) / "monitoring",
    "backtests": Path(CONFIG["output_root"]) / "backtests",
}
for _path in OUTPUT_DIRS.values():
    _path.mkdir(parents=True, exist_ok=True)


def get_scaler(name):
    name = name.lower().strip()
    if name == "standard":
        return StandardScaler()
    if name == "minmax":
        return MinMaxScaler()
    raise ValueError(f"Unsupported scaler: {name}")


def save_current_plot(name):
    plt.tight_layout()
    plt.savefig(OUTPUT_DIRS["plots"] / name, dpi=150, bbox_inches="tight")
    plt.close()


def load_files(path, limit, force_file=None):
    files = sorted(glob.glob(os.path.join(path, "*.txt")))[:limit]
    if force_file:
        forced_path = os.path.join(path, force_file)
        if os.path.exists(forced_path) and forced_path not in files:
            files.append(forced_path)

    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f)
            if df.empty:
                continue
            df["ticker"] = os.path.basename(f).split(".")[0].upper()
            dfs.append(df)
        except Exception:
            continue

    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


def add_trailing_zscore(df, col, window):
    rolling_mean = df[col].rolling(window).mean()
    rolling_std = df[col].rolling(window).std()
    return (df[col] - rolling_mean) / (rolling_std + 1e-9)


def create_features(df):
    df = df.sort_values("Date").copy()

    prev_close = df["Close"].shift(1)

    df["price_diff"] = df["Close"].diff()
    df["return"] = df["Close"].pct_change()
    df["log_return"] = np.log(df["Close"] / prev_close)
    df["return_5d"] = df["Close"].pct_change(5)
    df["return_10d"] = df["Close"].pct_change(10)

    df["ma5"] = df["Close"].rolling(5).mean()
    df["ma10"] = df["Close"].rolling(10).mean()
    df["ma20"] = df["Close"].rolling(20).mean()
    df["ma50"] = df["Close"].rolling(50).mean()

    df["ma_gap_5"] = df["Close"] / (df["ma5"] + 1e-9) - 1
    df["ma_gap_10"] = df["Close"] / (df["ma10"] + 1e-9) - 1
    df["ma_gap_20"] = df["Close"] / (df["ma20"] + 1e-9) - 1
    df["trend_strength"] = df["ma10"] / (df["ma20"] + 1e-9) - 1

    df["volatility_10"] = df["return"].rolling(10).std()
    df["volatility_20"] = df["return"].rolling(20).std()
    df["vol_regime"] = df["volatility_20"] / (df["volatility_20"].rolling(50).mean() + 1e-9)

    df["momentum_5"] = df["Close"] / (df["Close"].shift(5) + 1e-9) - 1
    df["momentum_10"] = df["Close"] / (df["Close"].shift(10) + 1e-9) - 1

    df["hl_range"] = (df["High"] - df["Low"]) / (df["Close"] + 1e-9)
    df["oc_gap"] = (df["Open"] - prev_close) / (prev_close + 1e-9)
    df["intraday_return"] = (df["Close"] - df["Open"]) / (df["Open"] + 1e-9)
    df["close_position"] = (df["Close"] - df["Low"]) / (df["High"] - df["Low"] + 1e-9)
    df["candlestick_body"] = (df["Close"] - df["Open"]) / (df["High"] - df["Low"] + 1e-9)

    for lag in [1, 2, 3, 5, 10]:
        df[f"return_lag_{lag}"] = df["return"].shift(lag)

    for lag in [1, 3, 5, 10]:
        df[f"volume_ratio_{lag}"] = df["Volume"] / (df["Volume"].shift(lag) + 1e-9)

    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / (loss + 1e-9)
    df["rsi_14"] = 100 - (100 / (1 + rs))

    ema_12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema_26 = df["Close"].ewm(span=26, adjust=False).mean()
    df["macd"] = ema_12 - ema_26
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
    df["macd_hist"] = df["macd"] - df["macd_signal"]

    rolling_low_14 = df["Low"].rolling(14).min()
    rolling_high_14 = df["High"].rolling(14).max()
    df["stoch_k"] = 100 * (df["Close"] - rolling_low_14) / (rolling_high_14 - rolling_low_14 + 1e-9)
    df["stoch_d"] = df["stoch_k"].rolling(3).mean()

    df["rel_volume"] = df["Volume"] / (df["Volume"].rolling(20).mean() + 1e-9)
    df["dollar_volume"] = df["Close"] * df["Volume"]
    df["dollar_volume_ma20"] = df["dollar_volume"].rolling(20).mean()
    df["dollar_volume_ratio"] = df["dollar_volume"] / (df["dollar_volume_ma20"] + 1e-9)

    rolling_std_20 = df["Close"].rolling(20).std()
    df["bb_mid"] = df["Close"].rolling(20).mean()
    df["bb_upper"] = df["bb_mid"] + 2 * rolling_std_20
    df["bb_lower"] = df["bb_mid"] - 2 * rolling_std_20
    df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / (df["bb_mid"] + 1e-9)
    df["bb_percent_b"] = (df["Close"] - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"] + 1e-9)

    true_range = pd.concat(
        [
            (df["High"] - df["Low"]),
            (df["High"] - prev_close).abs(),
            (df["Low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    df["atr_14"] = true_range.rolling(14).mean() / (df["Close"] + 1e-9)

    df["return_z_20"] = add_trailing_zscore(df, "return", 20)
    df["volatility_z_20"] = add_trailing_zscore(df, "volatility_20", 20)
    df["momentum_z_20"] = add_trailing_zscore(df, "momentum_10", 20)
    df["rel_volume_z_20"] = add_trailing_zscore(df, "rel_volume", 20)
    df["rsi_z_20"] = add_trailing_zscore(df, "rsi_14", 20)

    df["future_close"] = df["Close"].shift(-CONFIG["future_horizon"])
    df["future_return"] = df["future_close"] / df["Close"] - 1
    df["target_up"] = (df["future_close"] > df["Close"]).astype(int)
    return df


def add_market_features(data):
    if "SPY" not in data["ticker"].values:
        raise ValueError("SPY ticker not found. Market features require SPY.")

    spy = (
        data[data["ticker"] == "SPY"][["Date", "return", "future_return"]]
        .drop_duplicates("Date")
        .sort_values("Date")
        .rename(columns={"return": "market_return", "future_return": "market_future_return"})
    )
    spy["market_vol"] = spy["market_return"].rolling(20).std()
    spy["market_trend"] = spy["market_return"].rolling(50).mean()
    spy["market_regime"] = (spy["market_trend"] > 0).astype(int)
    spy["future_market_return"] = (
        spy["market_return"].rolling(CONFIG["future_horizon"]).sum().shift(-CONFIG["future_horizon"])
    )

    data = data.merge(
        spy[["Date", "market_return", "market_vol", "market_regime", "future_market_return", "market_future_return"]],
        on="Date",
        how="left",
    )
    data["market_return"] = data["market_return"].fillna(0)
    data["market_vol"] = data["market_vol"].fillna(data["market_vol"].median())
    data["market_regime"] = data["market_regime"].fillna(0).astype(int)
    data["future_market_return"] = data["future_market_return"].fillna(0)
    data["market_future_return"] = data["market_future_return"].fillna(0)
    data["future_excess_return"] = data["future_return"] - data["future_market_return"]
    return data


def add_cross_sectional_features(data, cols):
    data = data.copy()
    for col in cols:
        data[f"cs_rank_{col}"] = data.groupby("Date")[col].rank(pct=True)
    return data


def get_date_splits(dates, train_ratio=0.70, val_ratio=0.15):
    unique_dates = np.array(sorted(pd.to_datetime(pd.Series(dates)).unique()))
    train_end_idx = int(len(unique_dates) * train_ratio)
    val_end_idx = int(len(unique_dates) * (train_ratio + val_ratio))

    train_end_date = unique_dates[train_end_idx]
    val_end_date = unique_dates[val_end_idx]
    return train_end_date, val_end_date


def get_valid_sequence_end_indices(df, seq_len):
    valid_idx = []
    for _, grp in df.groupby("ticker", sort=False):
        idx = grp.index.to_list()
        valid_idx.extend(idx[seq_len - 1 :])
    return valid_idx


@dataclass
class SplitData:
    train_idx: list
    val_idx: list
    test_idx: list


def build_date_split_indices(df, seq_len):
    valid_idx = get_valid_sequence_end_indices(df, seq_len)
    train_cut, val_cut = get_date_splits(df["Date"].values)

    train_idx = [i for i in valid_idx if df.at[i, "Date"] < train_cut]
    val_idx = [i for i in valid_idx if train_cut <= df.at[i, "Date"] < val_cut]
    test_idx = [i for i in valid_idx if df.at[i, "Date"] >= val_cut]

    print(f"Train end date: {pd.Timestamp(train_cut).date()}")
    print(f"Validation end date: {pd.Timestamp(val_cut).date()}")
    print(f"Valid sequences -> train: {len(train_idx)}, val: {len(val_idx)}, test: {len(test_idx)}")
    return SplitData(train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)


class SeqDataset(Dataset):
    def __init__(self, X_scaled, y, indices, seq_len):
        self.X = torch.tensor(X_scaled, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.indices = indices
        self.seq_len = seq_len

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        end_idx = self.indices[idx]
        start_idx = end_idx - self.seq_len + 1
        return self.X[start_idx : end_idx + 1], self.y[end_idx]


class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])


def evaluate_classifier(y_true, probs, threshold, title):
    preds = (probs >= threshold).astype(int)
    roc_auc = roc_auc_score(y_true, probs)
    pr_auc = average_precision_score(y_true, probs)
    acc = accuracy_score(y_true, preds)
    precision = precision_score(y_true, preds, zero_division=0)
    recall = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    mcc = matthews_corrcoef(y_true, preds)

    print(f"\n==== {title} ====")
    print(
        f"Accuracy: {acc:.4f} | Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | F1: {f1:.4f}"
    )
    print(f"ROC AUC: {roc_auc:.4f} | PR AUC: {pr_auc:.4f} | MCC: {mcc:.4f}")
    print("\nClassification Report")
    print(classification_report(y_true, preds, digits=4))

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "mcc": mcc,
        "preds": preds,
    }


def evaluate_regressor(actual_close, predicted_close, actual_return, predicted_return, title):
    mae = mean_absolute_error(actual_close, predicted_close)
    rmse = np.sqrt(mean_squared_error(actual_close, predicted_close))
    mape = np.mean(np.abs((actual_close - predicted_close) / (actual_close + 1e-9)))
    r2 = r2_score(actual_close, predicted_close)
    directional_accuracy = np.mean(np.sign(actual_return) == np.sign(predicted_return))

    print(f"\n==== {title} ====")
    print(
        f"MAE: {mae:.4f} | RMSE: {rmse:.4f} | "
        f"MAPE: {mape:.4f} | R2: {r2:.4f} | Directional Accuracy: {directional_accuracy:.4f}"
    )
    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "mape": float(mape),
        "r2": float(r2),
        "directional_accuracy": float(directional_accuracy),
    }


def compute_regression_weights(y_true, pred_dict):
    scores = {}
    for name, preds in pred_dict.items():
        rmse = np.sqrt(mean_squared_error(y_true, preds))
        scores[name] = 1.0 / (rmse + 1e-9)

    total = sum(scores.values())
    return {name: value / total for name, value in scores.items()}


def compute_trading_metrics(returns, equity_curve):
    gross_profit = returns[returns > 0].sum()
    gross_loss = -returns[returns < 0].sum()
    drawdown = (equity_curve - equity_curve.cummax()) / (equity_curve.cummax() + 1e-9)
    return {
        "sharpe_ratio": float((returns.mean() / (returns.std() + 1e-9)) * np.sqrt(252)),
        "max_drawdown": float(drawdown.min()),
        "profit_factor": float(gross_profit / (gross_loss + 1e-9)),
        "win_rate": float((returns > 0).mean()),
        "cumulative_return": float(equity_curve.iloc[-1] / (equity_curve.iloc[0] + 1e-9) - 1),
    }


def psi(reference, current, bins=10):
    reference = pd.Series(reference).replace([np.inf, -np.inf], np.nan).dropna()
    current = pd.Series(current).replace([np.inf, -np.inf], np.nan).dropna()
    if reference.empty or current.empty:
        return 0.0

    edges = np.unique(np.nanpercentile(reference, np.linspace(0, 100, bins + 1)))
    if len(edges) <= 2:
        return 0.0

    ref_hist, _ = np.histogram(reference, bins=edges)
    cur_hist, _ = np.histogram(current, bins=edges)
    ref_pct = np.clip(ref_hist / max(ref_hist.sum(), 1), 1e-6, None)
    cur_pct = np.clip(cur_hist / max(cur_hist.sum(), 1), 1e-6, None)
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))


def find_best_threshold(y_true, probs, metric="f1"):
    best_thresh = 0.50
    best_score = -1
    upper = 0.95 if metric == "accuracy" else 0.75
    for t in np.linspace(0.25, upper, 71):
        preds = (probs >= t).astype(int)
        score = score_predictions(y_true, preds, metric)
        if score > best_score:
            best_score = score
            best_thresh = float(t)
    return best_thresh, best_score


def compute_model_weights(y_val, pred_dict):
    scores = {}
    for name, probs in pred_dict.items():
        auc = roc_auc_score(y_val, probs)
        pr_auc = average_precision_score(y_val, probs)
        scores[name] = max((auc - 0.50) + 0.5 * (pr_auc - y_val.mean()), 0)

    score_sum = sum(scores.values())
    if score_sum <= 0:
        weights = {k: 1 / len(scores) for k in scores}
    else:
        weights = {k: v / score_sum for k, v in scores.items()}

    print("\nValidation-based ensemble weights")
    for name, weight in weights.items():
        print(f"{name}: {weight:.4f}")
    return weights


def score_predictions(y_true, preds, metric):
    if metric == "f1":
        return f1_score(y_true, preds, zero_division=0)
    if metric == "accuracy":
        return accuracy_score(y_true, preds)
    if metric == "balanced_accuracy":
        return balanced_accuracy_score(y_true, preds)
    if metric == "mcc":
        return matthews_corrcoef(y_true, preds)
    raise ValueError(f"Unsupported threshold metric: {metric}")


def build_rank_ensemble(rows, pred_dict, weights):
    blend_df = rows[["Date"]].copy().reset_index(drop=True)
    ensemble_score = np.zeros(len(blend_df), dtype=float)
    for name, probs in pred_dict.items():
        ranks = pd.Series(probs).groupby(blend_df["Date"]).rank(pct=True)
        ensemble_score += weights[name] * ranks.values
    return ensemble_score


def build_weighted_probability_ensemble(pred_dict, weights):
    out = np.zeros(len(next(iter(pred_dict.values()))), dtype=float)
    for name, probs in pred_dict.items():
        out += weights[name] * np.asarray(probs)
    return out


def make_time_decay_weights(date_series, min_weight=0.35):
    unique_dates = pd.Series(sorted(pd.to_datetime(pd.Series(date_series)).unique()))
    date_to_pos = {date: idx for idx, date in enumerate(unique_dates)}
    positions = pd.to_datetime(pd.Series(date_series)).map(date_to_pos).astype(float)
    scaled = positions / max(len(unique_dates) - 1, 1)
    return min_weight + (1.0 - min_weight) * scaled


def optimize_blend_and_threshold(y_true, raw_probs, rank_probs, metric="accuracy"):
    best = {
        "alpha": 1.0,
        "threshold": 0.50,
        "score": -np.inf,
        "blended_probs": np.asarray(raw_probs),
    }
    for alpha in np.linspace(0.0, 1.0, 21):
        blended = alpha * np.asarray(raw_probs) + (1.0 - alpha) * np.asarray(rank_probs)
        thresh, score = find_best_threshold(y_true, blended, metric=metric)
        if score > best["score"]:
            best = {
                "alpha": float(alpha),
                "threshold": float(thresh),
                "score": float(score),
                "blended_probs": blended,
            }
    return best


def get_imbalance_settings(y):
    pos = float(np.sum(y))
    neg = float(len(y) - pos)
    ratio = neg / (pos + 1e-6)
    if CONFIG["threshold_metric"] == "accuracy":
        return {
            "lgb_class_weight": None,
            "xgb_scale_pos_weight": max(np.sqrt(ratio), 1.0),
            "lstm_pos_weight": max(np.sqrt(ratio), 1.0),
        }
    return {
        "lgb_class_weight": "balanced",
        "xgb_scale_pos_weight": ratio,
        "lstm_pos_weight": ratio,
    }


def make_loader(dataset, batch_size, shuffle):
    num_workers = 2 if CONFIG["device"] == "cuda" else 0
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=(CONFIG["device"] == "cuda"),
        persistent_workers=(num_workers > 0),
    )


def predict_lstm(model, loader):
    model.eval()
    out = []
    autocast_enabled = CONFIG["device"] == "cuda"
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(CONFIG["device"], non_blocking=True)
            with torch.cuda.amp.autocast(enabled=autocast_enabled):
                probs = torch.sigmoid(model(xb)).squeeze(1).cpu().numpy()
            out.extend(probs.tolist())
    return np.array(out)


def train_lstm_model(X_scaled_full, y_full, split_data, input_size):
    train_ds = SeqDataset(X_scaled_full, y_full, split_data.train_idx, CONFIG["seq_len"])
    val_ds = SeqDataset(X_scaled_full, y_full, split_data.val_idx, CONFIG["seq_len"])
    test_ds = SeqDataset(X_scaled_full, y_full, split_data.test_idx, CONFIG["seq_len"])

    train_loader = make_loader(train_ds, CONFIG["batch_size"], shuffle=True)
    val_loader = make_loader(val_ds, CONFIG["batch_size"], shuffle=False)
    test_loader = make_loader(test_ds, CONFIG["batch_size"], shuffle=False)

    model = LSTMModel(input_size=input_size).to(CONFIG["device"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    y_train_seq = y_full[split_data.train_idx]
    lstm_ratio = get_imbalance_settings(y_train_seq)["lstm_pos_weight"]
    pos_weight = torch.tensor([lstm_ratio], device=CONFIG["device"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    scaler_amp = torch.cuda.amp.GradScaler(enabled=(CONFIG["device"] == "cuda"))

    best_state = None
    best_val_auc = -np.inf
    patience = 5
    patience_counter = 0

    print("\nTraining LSTM")
    for epoch in range(CONFIG["epochs"]):
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(CONFIG["device"], non_blocking=True)
            yb = yb.to(CONFIG["device"], non_blocking=True).unsqueeze(1)
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(CONFIG["device"] == "cuda")):
                logits = model(xb)
                loss = criterion(logits, yb)

            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler_amp.step(optimizer)
            scaler_amp.update()
            total_loss += loss.item()

        val_probs = predict_lstm(model, val_loader)
        val_auc = roc_auc_score(y_full[split_data.val_idx], val_probs)
        avg_loss = total_loss / max(len(train_loader), 1)
        print(f"Epoch {epoch + 1:02d} | Train Loss: {avg_loss:.4f} | Val AUC: {val_auc:.4f}")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered for LSTM.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    val_probs = predict_lstm(model, val_loader)
    test_probs = predict_lstm(model, test_loader)
    return model, val_probs, test_probs


print("Loading stock files...")
stocks = load_files(CONFIG["stocks_path"], CONFIG["max_stock_files"])
print("Loading ETF files...")
etfs = load_files(CONFIG["etf_path"], CONFIG["max_etf_files"], force_file="spy.us.txt")

if stocks.empty and etfs.empty:
    raise ValueError("No input files were loaded. Check the Kaggle paths.")

raw_data = pd.concat([stocks, etfs], ignore_index=True)
raw_data["Date"] = pd.to_datetime(raw_data["Date"])
raw_data = raw_data.sort_values(["ticker", "Date"]).reset_index(drop=True)
raw_data["dollar_volume"] = raw_data["Close"] * raw_data["Volume"]

if "SPY" not in raw_data["ticker"].values:
    alt_spy = os.path.join(CONFIG["etf_path"], "spy.txt")
    if os.path.exists(alt_spy):
        spy_df = pd.read_csv(alt_spy)
        spy_df["Date"] = pd.to_datetime(spy_df["Date"])
        spy_df["ticker"] = "SPY"
        raw_data = pd.concat([raw_data, spy_df], ignore_index=True)
        raw_data = raw_data.sort_values(["ticker", "Date"]).reset_index(drop=True)

raw_train_cut, _ = get_date_splits(raw_data["Date"].values)
universe_liquidity = (
    raw_data[raw_data["Date"] < raw_train_cut]
    .groupby("ticker")["dollar_volume"]
    .mean()
    .sort_values(ascending=False)
)
top_tickers = universe_liquidity.head(CONFIG["top_n_tickers"]).index.tolist()
if "SPY" in raw_data["ticker"].values and "SPY" not in top_tickers:
    top_tickers.append("SPY")

data = raw_data[raw_data["ticker"].isin(top_tickers)].copy()
data = data.groupby("ticker", group_keys=False).apply(create_features).reset_index(drop=True)
data = add_market_features(data)
data = add_cross_sectional_features(
    data,
    cols=["return", "return_5d", "momentum_10", "rel_volume", "volatility_20", "rsi_14", "ma_gap_20", "macd_hist", "bb_percent_b"],
)

feature_cols = [
    "price_diff",
    "return",
    "log_return",
    "return_5d",
    "return_10d",
    "ma_gap_5",
    "ma_gap_10",
    "ma_gap_20",
    "trend_strength",
    "volatility_10",
    "volatility_20",
    "vol_regime",
    "momentum_5",
    "momentum_10",
    "hl_range",
    "oc_gap",
    "intraday_return",
    "close_position",
    "candlestick_body",
    "return_lag_1",
    "return_lag_2",
    "return_lag_3",
    "return_lag_5",
    "return_lag_10",
    "volume_ratio_1",
    "volume_ratio_3",
    "volume_ratio_5",
    "volume_ratio_10",
    "rsi_14",
    "macd",
    "macd_signal",
    "macd_hist",
    "stoch_k",
    "stoch_d",
    "bb_width",
    "bb_percent_b",
    "atr_14",
    "rel_volume",
    "dollar_volume_ratio",
    "return_z_20",
    "volatility_z_20",
    "momentum_z_20",
    "rel_volume_z_20",
    "rsi_z_20",
    "market_return",
    "market_vol",
    "market_future_return",
    "market_regime",
    "cs_rank_return",
    "cs_rank_return_5d",
    "cs_rank_momentum_10",
    "cs_rank_rel_volume",
    "cs_rank_volatility_20",
    "cs_rank_rsi_14",
    "cs_rank_ma_gap_20",
    "cs_rank_macd_hist",
    "cs_rank_bb_percent_b",
]

data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=feature_cols + ["future_return", "future_close", "target_up"]).reset_index(drop=True)
data = data.sort_values(["ticker", "Date"]).reset_index(drop=True)

print(f"Filtered dataset shape: {data.shape}")
print(f"Tickers used: {sorted(data['ticker'].unique().tolist())}")
print(f"Positive class rate: {data['target_up'].mean():.4f}")

split_data = build_date_split_indices(data, CONFIG["seq_len"])

train_rows = data.loc[split_data.train_idx].copy().reset_index(drop=True)
val_rows = data.loc[split_data.val_idx].copy().reset_index(drop=True)
test_rows = data.loc[split_data.test_idx].copy().reset_index(drop=True)

X_train = train_rows[feature_cols]
y_train = train_rows["target_up"].astype(int)
X_val = val_rows[feature_cols]
y_val = val_rows["target_up"].astype(int)
X_test = test_rows[feature_cols]
y_test = test_rows["target_up"].astype(int)

print(f"Train rows: {len(X_train)}, Val rows: {len(X_val)}, Test rows: {len(X_test)}")
print(f"Feature count: {len(feature_cols)}")

scaler = get_scaler(CONFIG["scaler_type"])
scaler.fit(X_train)
X_scaled_full = scaler.transform(data[feature_cols])
y_full = data["target_up"].values.astype(np.float32)
train_sample_weights = make_time_decay_weights(train_rows["Date"])
imbalance_settings = get_imbalance_settings(y_train.values)


def lgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 250, 900),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 120),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 5.0, log=True),
        "random_state": CONFIG["random_seed"],
        "objective": "binary",
        "verbosity": -1,
        "class_weight": imbalance_settings["lgb_class_weight"],
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train,
        y_train,
        sample_weight=train_sample_weights,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
    )
    val_probs = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, val_probs)


print("\nOptimizing LightGBM with Optuna...")
study = optuna.create_study(direction="maximize")
study.optimize(lgb_objective, n_trials=CONFIG["num_trials"], show_progress_bar=False)
best_lgb_params = study.best_params
best_lgb_params.update(
    {
        "random_state": CONFIG["random_seed"],
        "objective": "binary",
        "verbosity": -1,
        "class_weight": imbalance_settings["lgb_class_weight"],
    }
)
print("Best LightGBM params:", best_lgb_params)

lgb_model = lgb.LGBMClassifier(**best_lgb_params)
lgb_model.fit(
    X_train,
    y_train,
    sample_weight=train_sample_weights,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
)
lgb_val_probs = lgb_model.predict_proba(X_val)[:, 1]
lgb_test_probs = lgb_model.predict_proba(X_test)[:, 1]

pos = y_train.sum()
neg = len(y_train) - pos
xgb_model = xgb.XGBClassifier(
    n_estimators=700,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=float(imbalance_settings["xgb_scale_pos_weight"]),
    objective="binary:logistic",
    eval_metric="auc",
    random_state=CONFIG["random_seed"],
    tree_method="hist",
)
print("\nTraining XGBoost...")
xgb_model.fit(
    X_train,
    y_train,
    sample_weight=train_sample_weights,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
xgb_val_probs = xgb_model.predict_proba(X_val)[:, 1]
xgb_test_probs = xgb_model.predict_proba(X_test)[:, 1]

y_train_reg = train_rows["future_return"].astype(np.float32)
y_val_reg = val_rows["future_return"].astype(np.float32)
y_test_reg = test_rows["future_return"].astype(np.float32)


def lgb_reg_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 250, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "num_leaves": trial.suggest_int("num_leaves", 24, 160),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 2.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 5.0, log=True),
        "random_state": CONFIG["random_seed"],
        "objective": "regression",
        "verbosity": -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train,
        y_train_reg,
        sample_weight=train_sample_weights,
        eval_set=[(X_val, y_val_reg)],
        eval_metric="l2",
        callbacks=[lgb.early_stopping(40), lgb.log_evaluation(0)],
    )
    preds = model.predict(X_val)
    return -np.sqrt(mean_squared_error(y_val_reg, preds))


print("\nOptimizing LightGBM regressor with Optuna...")
reg_study = optuna.create_study(direction="maximize")
reg_study.optimize(lgb_reg_objective, n_trials=CONFIG["regression_trials"], show_progress_bar=False)
best_lgb_reg_params = reg_study.best_params
best_lgb_reg_params.update(
    {
        "random_state": CONFIG["random_seed"],
        "objective": "regression",
        "verbosity": -1,
    }
)
lgb_reg_model = lgb.LGBMRegressor(**best_lgb_reg_params)
lgb_reg_model.fit(
    X_train,
    y_train_reg,
    sample_weight=train_sample_weights,
    eval_set=[(X_val, y_val_reg)],
    eval_metric="l2",
    callbacks=[lgb.early_stopping(40), lgb.log_evaluation(0)],
)
xgb_reg_model = xgb.XGBRegressor(
    n_estimators=700,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.05,
    reg_lambda=1.0,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=CONFIG["random_seed"],
    tree_method="hist",
)
print("\nTraining XGBoost regressor...")
xgb_reg_model.fit(
    X_train,
    y_train_reg,
    sample_weight=train_sample_weights,
    eval_set=[(X_val, y_val_reg)],
    verbose=False,
)

lgb_reg_val = lgb_reg_model.predict(X_val)
lgb_reg_test = lgb_reg_model.predict(X_test)
xgb_reg_val = xgb_reg_model.predict(X_val)
xgb_reg_test = xgb_reg_model.predict(X_test)

lstm_model, lstm_val_probs, lstm_test_probs = train_lstm_model(
    X_scaled_full=X_scaled_full,
    y_full=y_full,
    split_data=split_data,
    input_size=len(feature_cols),
)

single_model_val = {
    "LightGBM": lgb_val_probs,
    "XGBoost": xgb_val_probs,
    "LSTM": lstm_val_probs,
}
single_model_test = {
    "LightGBM": lgb_test_probs,
    "XGBoost": xgb_test_probs,
    "LSTM": lstm_test_probs,
}

print("\nValidation metrics by model")
for name, probs in single_model_val.items():
    print(
        f"{name}: ROC AUC={roc_auc_score(y_val, probs):.4f}, "
        f"PR AUC={average_precision_score(y_val, probs):.4f}"
    )

ensemble_weights = compute_model_weights(y_val, single_model_val)

val_prob_ensemble = build_weighted_probability_ensemble(single_model_val, ensemble_weights)
test_prob_ensemble = build_weighted_probability_ensemble(single_model_test, ensemble_weights)
val_rank_ensemble = build_rank_ensemble(val_rows, single_model_val, ensemble_weights)
test_rank_ensemble = build_rank_ensemble(test_rows, single_model_test, ensemble_weights)

blend_result = optimize_blend_and_threshold(
    y_true=y_val,
    raw_probs=val_prob_ensemble,
    rank_probs=val_rank_ensemble,
    metric=CONFIG["threshold_metric"],
)
blend_alpha = blend_result["alpha"]
best_thresh = blend_result["threshold"]
best_threshold_score = blend_result["score"]
val_ensemble_probs = blend_result["blended_probs"]
test_ensemble_probs = blend_alpha * test_prob_ensemble + (1.0 - blend_alpha) * test_rank_ensemble

best_thresh_f1, best_val_f1 = find_best_threshold(y_val, val_ensemble_probs, metric="f1")
best_thresh_acc, best_val_acc = find_best_threshold(y_val, val_ensemble_probs, metric="accuracy")
print(
    f"\nSelected validation threshold: {best_thresh:.4f} | "
    f"{CONFIG['threshold_metric'].upper()} score: {best_threshold_score:.4f}"
)
print(f"Best raw/rank blend alpha (raw weight): {blend_alpha:.2f}")
print(f"Validation F1-optimal threshold: {best_thresh_f1:.4f} | F1: {best_val_f1:.4f}")
print(f"Validation Accuracy-optimal threshold: {best_thresh_acc:.4f} | Accuracy: {best_val_acc:.4f}")

test_metrics = evaluate_classifier(
    y_true=y_test,
    probs=test_ensemble_probs,
    threshold=best_thresh,
    title="TEST SET METRICS",
)
test_preds = test_metrics["preds"]

regression_val_preds = {
    "LightGBMRegressor": lgb_reg_val,
    "XGBoostRegressor": xgb_reg_val,
}
regression_test_preds = {
    "LightGBMRegressor": lgb_reg_test,
    "XGBoostRegressor": xgb_reg_test,
}
regression_weights = compute_regression_weights(y_val_reg, regression_val_preds)
val_reg_ensemble = build_weighted_probability_ensemble(regression_val_preds, regression_weights)
test_reg_ensemble = build_weighted_probability_ensemble(regression_test_preds, regression_weights)
predicted_next_close = test_rows["Close"].values * (1 + test_reg_ensemble)
regression_metrics = evaluate_regressor(
    actual_close=test_rows["future_close"].values,
    predicted_close=predicted_next_close,
    actual_return=y_test_reg.values,
    predicted_return=test_reg_ensemble,
    title="NEXT-DAY CLOSE REGRESSION METRICS",
)

plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_test, test_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
save_current_plot("confusion_matrix.png")

fpr, tpr, _ = roc_curve(y_test, test_ensemble_probs)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, test_ensemble_probs)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, test_ensemble_probs):.4f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(recall_curve, precision_curve, label=f"PR AUC = {average_precision_score(y_test, test_ensemble_probs):.4f}")
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
save_current_plot("classification_curves.png")

importances = pd.Series(lgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(20)
plt.figure(figsize=(10, 7))
sns.barplot(x=importances.values, y=importances.index, orient="h", palette="viridis")
plt.title("Top 20 LightGBM Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
save_current_plot("feature_importance.png")

regression_plot_df = test_rows[["Date", "ticker", "future_close"]].copy()
regression_plot_df["predicted_next_close"] = predicted_next_close
focus_ticker = regression_plot_df["ticker"].value_counts().index[0]
focus_regression = regression_plot_df[regression_plot_df["ticker"] == focus_ticker].tail(80)

plt.figure(figsize=(12, 5))
plt.plot(focus_regression["Date"], focus_regression["future_close"], label=f"{focus_ticker} Actual Next Close")
plt.plot(focus_regression["Date"], focus_regression["predicted_next_close"], label=f"{focus_ticker} Predicted Next Close")
plt.legend()
plt.title("Next-Day Closing Price Prediction")
plt.xlabel("Date")
plt.ylabel("Price")
save_current_plot("next_close_prediction.png")

test_bt = test_rows.copy()
test_bt["prob"] = test_ensemble_probs
test_bt["pred"] = test_preds
test_bt = test_bt.sort_values(["Date", "ticker"]).reset_index(drop=True)

test_bt["rank"] = test_bt.groupby("Date")["prob"].rank(pct=True)
test_bt["target_weight"] = (test_bt["rank"] >= 0.70).astype(float)
test_bt["target_weight"] = test_bt.groupby("Date")["target_weight"].transform(
    lambda x: np.where(x.sum() == 0, 0, x / x.sum())
)
test_bt["prev_weight"] = test_bt.groupby("ticker")["target_weight"].shift(1).fillna(0)
test_bt["weight_change"] = (test_bt["target_weight"] - test_bt["prev_weight"]).abs()
test_bt["position_ret"] = test_bt["prev_weight"] * test_bt["future_return"]

simple_port_df = (
    test_bt.groupby("Date")
    .agg(
        portfolio_ret=("position_ret", "sum"),
        turnover=("weight_change", "sum"),
        market_ret=("market_future_return", "first"),
    )
    .reset_index()
)
simple_port_df["strategy_return"] = simple_port_df["portfolio_ret"] - (
    simple_port_df["turnover"] * CONFIG["transaction_cost"]
)
simple_port_df["cum_market"] = (1 + simple_port_df["market_ret"]).cumprod()
simple_port_df["cum_strategy"] = (1 + simple_port_df["strategy_return"]).cumprod()
simple_trade_metrics = compute_trading_metrics(simple_port_df["strategy_return"], simple_port_df["cum_strategy"])

simple_ret = simple_port_df["strategy_return"]
simple_sharpe = (simple_ret.mean() / (simple_ret.std() + 1e-9)) * np.sqrt(252)
simple_sortino = (
    simple_ret.mean() / (simple_ret[simple_ret < 0].std() + 1e-9)
) * np.sqrt(252)
simple_drawdown = (
    simple_port_df["cum_strategy"] - simple_port_df["cum_strategy"].cummax()
) / (simple_port_df["cum_strategy"].cummax() + 1e-9)

print("\n==== SIMPLE LONG-ONLY STRATEGY METRICS ====")
print(
    f"Sharpe: {simple_sharpe:.2f} | Sortino: {simple_sortino:.2f} | "
    f"Max Drawdown: {simple_drawdown.min():.2%}"
)
print(
    f"Profit Factor: {simple_trade_metrics['profit_factor']:.2f} | "
    f"Win Rate: {simple_trade_metrics['win_rate']:.2%} | "
    f"Cumulative Return: {simple_trade_metrics['cumulative_return']:.2%}"
)

plt.figure(figsize=(10, 5))
plt.plot(simple_port_df["Date"], simple_port_df["cum_market"], label="SPY Market")
plt.plot(simple_port_df["Date"], simple_port_df["cum_strategy"], label="Long-Only Strategy")
plt.legend()
plt.title("Long-Only Strategy vs Market")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
save_current_plot("long_only_equity_curve.png")

rank_df = test_rows.copy().sort_values(["Date", "ticker"]).reset_index(drop=True)
rank_df["prob"] = test_ensemble_probs
rank_df["rank"] = rank_df.groupby("Date")["prob"].rank(pct=True)
rank_df["long_signal"] = (rank_df["rank"] >= 0.85).astype(int)
rank_df["short_signal"] = -(rank_df["rank"] <= 0.15).astype(int)
rank_df["target_weight"] = rank_df["long_signal"] + rank_df["short_signal"]
rank_df["target_weight"] = rank_df.groupby("ticker")["target_weight"].transform(
    lambda x: x.ewm(span=5, adjust=False).mean()
)

gross_exposure = rank_df.groupby("Date")["target_weight"].transform(lambda x: x.abs().sum())
rank_df["adj_weight"] = np.where(gross_exposure == 0, 0, rank_df["target_weight"] / gross_exposure)
rank_df["lagged_weight"] = rank_df.groupby("ticker")["adj_weight"].shift(1).fillna(0)
rank_df["prev_weight"] = rank_df.groupby("ticker")["adj_weight"].shift(1).fillna(0)
rank_df["weight_change"] = (rank_df["adj_weight"] - rank_df["prev_weight"]).abs()
rank_df["position_ret"] = rank_df["lagged_weight"] * rank_df["future_return"]

port_df = (
    rank_df.groupby("Date")
    .agg(
        portfolio_ret=("position_ret", "sum"),
        turnover=("weight_change", "sum"),
        market_ret=("market_future_return", "first"),
    )
    .reset_index()
)

target_vol = 0.02
port_df["rolling_vol"] = port_df["portfolio_ret"].rolling(20).std().shift(1)
port_df["vol_scale"] = target_vol / (port_df["rolling_vol"] + 1e-9)
port_df["vol_scale"] = port_df["vol_scale"].clip(upper=2.5)
port_df["strategy_return"] = port_df["portfolio_ret"] * port_df["vol_scale"]
port_df["strategy_return"] -= (
    port_df["turnover"] * port_df["vol_scale"] * CONFIG["transaction_cost"]
)
port_df = port_df.dropna().reset_index(drop=True)

capital = 100000
port_df["equity_curve"] = capital * (1 + port_df["strategy_return"]).cumprod()
port_df["market_curve"] = capital * (1 + port_df["market_ret"]).cumprod()

ret_series = port_df["strategy_return"]
sharpe_adv = (ret_series.mean() / (ret_series.std() + 1e-9)) * np.sqrt(252)
sortino_adv = (
    ret_series.mean() / (ret_series[ret_series < 0].std() + 1e-9)
) * np.sqrt(252)
drawdown_series = (
    port_df["equity_curve"] - port_df["equity_curve"].cummax()
) / (port_df["equity_curve"].cummax() + 1e-9)
advanced_trade_metrics = compute_trading_metrics(port_df["strategy_return"], port_df["equity_curve"])

print("\n==== ADVANCED LONG-SHORT STRATEGY METRICS ====")
print(f"Average daily portfolio return: {port_df['portfolio_ret'].mean():.6f}")
print(f"Average daily turnover: {port_df['turnover'].mean():.4f}")
print(f"Average leverage: {port_df['vol_scale'].mean():.2f}x")
print(f"Max leverage: {port_df['vol_scale'].max():.2f}x")
print(f"Advanced Sharpe: {sharpe_adv:.2f}")
print(f"Advanced Sortino: {sortino_adv:.2f}")
print(f"Advanced Max Drawdown: {drawdown_series.min():.2%}")
print(
    f"Advanced Profit Factor: {advanced_trade_metrics['profit_factor']:.2f} | "
    f"Win Rate: {advanced_trade_metrics['win_rate']:.2%} | "
    f"Cumulative Return: {advanced_trade_metrics['cumulative_return']:.2%}"
)

plt.figure(figsize=(10, 5))
plt.plot(port_df["Date"], port_df["equity_curve"], label="Advanced Vol-Targeted Strategy")
plt.plot(port_df["Date"], port_df["market_curve"], label="SPY Market", alpha=0.8)
plt.legend()
plt.title("Advanced Long-Short Strategy Equity Curve")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")
save_current_plot("long_short_equity_curve.png")

plt.figure(figsize=(10, 4))
plt.fill_between(port_df["Date"], drawdown_series.values, 0, color="crimson", alpha=0.35)
plt.title("Strategy Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown")
save_current_plot("strategy_drawdown.png")

artifacts = {
    "feature_cols": feature_cols,
    "threshold": best_thresh,
    "threshold_metric": CONFIG["threshold_metric"],
    "blend_alpha": blend_alpha,
    "ensemble_weights": ensemble_weights,
    "regression_weights": regression_weights,
    "scaler": scaler,
    "lgb_val_auc": roc_auc_score(y_val, lgb_val_probs),
    "xgb_val_auc": roc_auc_score(y_val, xgb_val_probs),
    "lstm_val_auc": roc_auc_score(y_val, lstm_val_probs),
    "regression_metrics": regression_metrics,
    "simple_trade_metrics": simple_trade_metrics,
    "advanced_trade_metrics": advanced_trade_metrics,
}

pickle.dump(lgb_model, open(OUTPUT_DIRS["models"] / "lgb_model.pkl", "wb"))
pickle.dump(xgb_model, open(OUTPUT_DIRS["models"] / "xgb_model.pkl", "wb"))
pickle.dump(lgb_reg_model, open(OUTPUT_DIRS["models"] / "lgb_reg_model.pkl", "wb"))
pickle.dump(xgb_reg_model, open(OUTPUT_DIRS["models"] / "xgb_reg_model.pkl", "wb"))
torch.save(lstm_model.state_dict(), OUTPUT_DIRS["models"] / "lstm_model.pth")
pickle.dump(artifacts, open(OUTPUT_DIRS["models"] / "model_artifacts.pkl", "wb"))
print("\nModels and metadata saved successfully.")


def predict_signal(preprocessed_df):
    """
    Generate an ensemble signal from already engineered features.
    Input must contain all `feature_cols` and the last CONFIG['seq_len'] rows
    must belong to the same ticker.
    """
    if len(preprocessed_df) < CONFIG["seq_len"]:
        raise ValueError(f"Need at least {CONFIG['seq_len']} rows for prediction.")

    missing_cols = sorted(set(feature_cols) - set(preprocessed_df.columns))
    if missing_cols:
        raise ValueError(f"Missing features: {missing_cols}")

    if "ticker" in preprocessed_df.columns and preprocessed_df["ticker"].iloc[-CONFIG["seq_len"] :].nunique() > 1:
        raise ValueError("Last sequence spans multiple tickers.")

    X_new = preprocessed_df[feature_cols].copy()
    X_last = X_new.iloc[[-1]]
    X_scaled_seq = scaler.transform(X_new)

    lgb_prob = float(lgb_model.predict_proba(X_last)[:, 1][0])
    xgb_prob = float(xgb_model.predict_proba(X_last)[:, 1][0])

    lstm_model.eval()
    seq_tensor = (
        torch.tensor(X_scaled_seq[-CONFIG["seq_len"] :], dtype=torch.float32)
        .unsqueeze(0)
        .to(CONFIG["device"])
    )
    with torch.no_grad():
        lstm_prob = float(torch.sigmoid(lstm_model(seq_tensor)).cpu().numpy()[0][0])

    final_prob = (
        ensemble_weights["LightGBM"] * lgb_prob
        + ensemble_weights["XGBoost"] * xgb_prob
        + ensemble_weights["LSTM"] * lstm_prob
    )
    lgb_reg_return = float(lgb_reg_model.predict(X_last)[0])
    xgb_reg_return = float(xgb_reg_model.predict(X_last)[0])
    predicted_return = (
        regression_weights["LightGBMRegressor"] * lgb_reg_return
        + regression_weights["XGBoostRegressor"] * xgb_reg_return
    )
    predicted_next_close = float(preprocessed_df["Close"].iloc[-1] * (1 + predicted_return))
    return {
        "probability": float(final_prob),
        "signal": int(final_prob >= best_thresh),
        "predicted_next_return": float(predicted_return),
        "predicted_next_close": predicted_next_close,
        "lgb_prob": lgb_prob,
        "xgb_prob": xgb_prob,
        "lstm_prob": lstm_prob,
    }


feature_drift = pd.DataFrame(
    [{"feature": col, "psi": psi(train_rows[col], test_rows[col])} for col in feature_cols]
).sort_values("psi", ascending=False)
prediction_drift = psi(val_ensemble_probs, test_ensemble_probs)

monitoring_summary = {
    "prediction_drift_psi": prediction_drift,
    "mean_feature_psi": float(feature_drift["psi"].mean()),
    "features_above_0_2_psi": int((feature_drift["psi"] > 0.2).sum()),
    "validation_accuracy": float(accuracy_score(y_val, (val_ensemble_probs >= best_thresh).astype(int))),
    "test_accuracy": float(test_metrics["accuracy"]),
    "accuracy_drop": float(
        accuracy_score(y_val, (val_ensemble_probs >= best_thresh).astype(int)) - test_metrics["accuracy"]
    ),
}
feature_drift.to_csv(OUTPUT_DIRS["monitoring"] / "feature_drift.csv", index=False)
json.dump(monitoring_summary, open(OUTPUT_DIRS["monitoring"] / "monitoring_summary.json", "w"), indent=2)
json.dump(
    {
        "dashboard": "stock_pipeline_monitoring",
        "metrics": monitoring_summary,
        "top_drift_features": feature_drift.head(10).to_dict(orient="records"),
    },
    open(OUTPUT_DIRS["monitoring"] / "grafana_metrics.json", "w"),
    indent=2,
)

if Report is not None and DataDriftPreset is not None and ClassificationPreset is not None:
    try:
        evidently_report = Report(metrics=[DataDriftPreset(), ClassificationPreset()])
        evidently_report.run(
            reference_data=train_rows[feature_cols + ["target_up"]].sample(min(len(train_rows), 5000), random_state=CONFIG["random_seed"]),
            current_data=test_rows[feature_cols + ["target_up"]].sample(min(len(test_rows), 5000), random_state=CONFIG["random_seed"]),
        )
        evidently_report.save_html(str(OUTPUT_DIRS["monitoring"] / "evidently_report.html"))
    except Exception as exc:
        print(f"Evidently report skipped: {exc}")

test_predictions = test_rows[["Date", "ticker", "Close", "future_close", "future_return"]].copy()
test_predictions["probability_up"] = test_ensemble_probs
test_predictions["predicted_signal"] = test_preds
test_predictions["predicted_next_close"] = predicted_next_close
test_predictions["predicted_next_return"] = test_reg_ensemble
test_predictions.to_csv(OUTPUT_DIRS["predictions"] / "test_predictions.csv", index=False)

latest_predictions = []
for ticker, grp in data.groupby("ticker", sort=False):
    grp = grp.dropna(subset=feature_cols).sort_values("Date")
    if len(grp) < CONFIG["seq_len"]:
        continue
    latest_predictions.append(
        {
            "ticker": ticker,
            "date": grp["Date"].iloc[-1],
            "current_close": float(grp["Close"].iloc[-1]),
            **predict_signal(grp.tail(CONFIG["seq_len"])),
        }
    )
latest_predictions_df = pd.DataFrame(latest_predictions)
if not latest_predictions_df.empty:
    latest_predictions_df = latest_predictions_df.sort_values("probability", ascending=False).reset_index(drop=True)
latest_predictions_df.to_csv(OUTPUT_DIRS["predictions"] / "latest_predictions.csv", index=False)

if not latest_predictions_df.empty:
    plt.figure(figsize=(12, 6))
    top_latest = latest_predictions_df.head(15)
    sns.barplot(data=top_latest, x="probability", y="ticker", orient="h", palette="crest")
    plt.title("Top Latest Upward Probabilities")
    plt.xlabel("Probability")
    plt.ylabel("Ticker")
    save_current_plot("latest_prediction_snapshot.png")

if shap is not None:
    try:
        shap_sample = train_rows[feature_cols].sample(min(len(train_rows), 2000), random_state=CONFIG["random_seed"])
        shap_explainer = shap.TreeExplainer(lgb_model)
        shap_values = shap_explainer.shap_values(shap_sample)
        if isinstance(shap_values, list):
            shap_values = shap_values[-1]
        shap_importance = (
            pd.DataFrame({"feature": feature_cols, "mean_abs_shap": np.abs(shap_values).mean(axis=0)})
            .sort_values("mean_abs_shap", ascending=False)
        )
        shap_importance.to_csv(OUTPUT_DIRS["reports"] / "shap_feature_importance.csv", index=False)
    except Exception as exc:
        print(f"SHAP generation skipped: {exc}")

json.dump(
    {
        "classification_test": {k: v for k, v in test_metrics.items() if k != "preds"},
        "regression_test": regression_metrics,
        "long_only_trading": simple_trade_metrics,
        "long_short_trading": advanced_trade_metrics,
        "monitoring": monitoring_summary,
        "latest_predictions_preview": latest_predictions_df.head(10).to_dict(orient="records"),
    },
    open(OUTPUT_DIRS["reports"] / "metrics_report.json", "w"),
    indent=2,
    default=str,
)


Device: cpu
Loading stock files...
Loading ETF files...
Filtered dataset shape: (180276, 79)
Tickers used: ['A', 'AA', 'AABA', 'AAPL', 'ABC', 'ABT', 'ABX', 'ADBE', 'ADI', 'ADP', 'ADSK', 'AEE', 'AES', 'AGN', 'AIG', 'ALL', 'AMAT', 'AMD', 'AMGN', 'AMZN', 'AN', 'ANTM', 'APA', 'APC', 'SPY']
Positive class rate: 0.4814
Train end date: 2003-07-24
Validation end date: 2010-09-17
Valid sequences -> train: 90241, val: 44536, test: 45024
Train rows: 90241, Val rows: 44536, Test rows: 45024
Feature count: 57


[I 2026-03-25 04:03:11,965] A new study created in memory with name: no-name-a1b87f29-9617-4bc4-ad79-d943d2262fcc



Optimizing LightGBM with Optuna...
Training until validation scores don't improve for 50 rounds


[I 2026-03-25 04:03:14,694] Trial 0 finished with value: 0.5118940364786446 and parameters: {'n_estimators': 859, 'learning_rate': 0.03883688980035981, 'max_depth': 8, 'num_leaves': 46, 'min_child_samples': 115, 'subsample': 0.9014581796311161, 'colsample_bytree': 0.9443815996989764, 'reg_alpha': 0.000968772367217973, 'reg_lambda': 1.4019731203981143}. Best is trial 0 with value: 0.5118940364786446.


Early stopping, best iteration is:
[6]	valid_0's auc: 0.508014	valid_0's binary_logloss: 0.693024
Training until validation scores don't improve for 50 rounds


...

Early stopping, best iteration is:
[77]	valid_0's l2: 0.000677836
Training until validation scores don't improve for 40 rounds


[I 2026-03-25 04:05:02,823] Trial 10 finished with value: -0.026077684518506502 and parameters: {'n_estimators': 798, 'learning_rate': 0.03740767770898258, 'max_depth': 3, 'num_leaves': 84, 'min_child_samples': 20, 'subsample': 0.8909894506411511, 'colsample_bytree': 0.7235148438008545, 'reg_alpha': 0.0017095486554035589, 'reg_lambda': 3.9231465218899784}. Best is trial 9 with value: -0.0260352784067773.


Early stopping, best iteration is:
[99]	valid_0's l2: 0.000680046
Training until validation scores don't improve for 40 rounds


[I 2026-03-25 04:05:06,020] Trial 11 finished with value: -0.02605844343898797 and parameters: {'n_estimators': 653, 'learning_rate': 0.024027563930365044, 'max_depth': 5, 'num_leaves': 28, 'min_child_samples': 45, 'subsample': 0.8380960373822609, 'colsample_bytree': 0.9816862860710351, 'reg_alpha': 0.001739822902779107, 'reg_lambda': 0.0014721190870577194}. Best is trial 9 with value: -0.0260352784067773.


Early stopping, best iteration is:
[108]	valid_0's l2: 0.000679042
Training until validation scores don't improve for 40 rounds
Early stopping, best iteration is:
[77]	valid_0's l2: 0.000677836

Training XGBoost regressor...

Training LSTM
Epoch 01 | Train Loss: 0.7591 | Val AUC: 0.5082
Epoch 02 | Train Loss: 0.7541 | Val AUC: 0.5098
Epoch 03 | Train Loss: 0.7525 | Val AUC: 0.5142
Epoch 04 | Train Loss: 0.7516 | Val AUC: 0.5174
Epoch 05 | Train Loss: 0.7495 | Val AUC: 0.5158
Epoch 06 | Train Loss: 0.7481 | Val AUC: 0.5174
Epoch 07 | Train Loss: 0.7469 | Val AUC: 0.5164
Epoch 08 | Train Loss: 0.7454 | Val AUC: 0.5191
Epoch 09 | Train Loss: 0.7438 | Val AUC: 0.5200
Epoch 10 | Train Loss: 0.7413 | Val AUC: 0.5169
Epoch 11 | Train Loss: 0.7391 | Val AUC: 0.5188
Epoch 12 | Train Loss: 0.7361 | Val AUC: 0.5207
Epoch 13 | Train Loss: 0.7336 | Val AUC: 0.5170
Epoch 14 | Train Loss: 0.7308 | Val AUC: 0.5177
Epoch 15 | Train Loss: 0.7271 | Val AUC: 0.5170
Epoch 16 | Train Loss: 0.7243 | Val AUC: